In [29]:
import os
import json
import sqlite3
import torch
import ollama
import gradio as gr
from PIL import Image
from diffusers import AutoPipelineForText2Image
from kokoro import KPipeline
import soundfile as sf
from typing import Any, Dict, List
import numpy as np

In [20]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Running models on device: {device}")

LLM_MODEL = "qwen3:8b"
DB = "prices.db"

Running models on device: mps


In [ ]:
# A. Local Image Model (SDXL-Turbo) Optimization
print("Loading SDXL-Turbo Model...")
image_pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", 
    torch_dtype=torch.float16 if device == "mps" else torch.float32, 
    variant="fp16" if device == "mps" else None
)

if device == "mps":
    image_pipe.enable_attention_slicing()

Loading SDXL-Turbo Model...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

In [22]:
print ("Loading kokoro TTS Engine...")
tts_pipeline = KPipeline(lang_code="a")


Loading kokoro TTS Engine...


/Users/user/Downloads/llm-engineering-hands-on/venv/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/Users/user/Downloads/llm-engineering-hands-on/venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [23]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [39]:
import sqlite3

DB = "prices.db"

# 1. Database table creation and sample data population
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # Create the prices table if it does not exist
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS prices (
            city TEXT PRIMARY KEY,
            price INTEGER
        )
    ''')
    
    # Insert initial sample flight prices
    sample_data = [
        ('paris', 450),
        ('london', 500),
        ('new york', 750),
        ('tokyo', 850),
        ('dubai', 400)
    ]
    cursor.executemany('INSERT OR REPLACE INTO prices (city, price) VALUES (?, ?)', sample_data)
    conn.commit()

print("✅ 'prices' table successfully created and populated!")

✅ 'prices' table successfully created and populated!


In [40]:
def get_ticket_price(city: str) -> str:
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [25]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"]
    }
}
tools = [{"type": "function", "function": price_function}]

In [30]:
from typing import Any

def handle_tool_calls_and_return_cities(message: Any) -> tuple[list[dict[str, Any]], list[str]]:
    responses: list[dict[str, Any]] = []
    cities: list[str] = []
    
    if hasattr(message, 'tool_calls') and message.tool_calls:
        for tool_call in message.tool_calls:
            if tool_call.function.name == "get_ticket_price":
                arguments = tool_call.function.arguments
                if isinstance(arguments, str):
                    arguments = json.loads(arguments)
                
                city = arguments.get('destination_city')
                if city:
                    cities.append(city)
                    price_details = get_ticket_price(city)
                    responses.append({
                        "role": "tool",
                        "content": price_details
                    })
    return responses, cities

In [31]:
def artist(city: str):
    prompt = f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style"
    image = image_pipe(prompt=prompt, num_inference_steps=2, guidance_scale=0.0).images[0]
    return image



In [32]:
def talker(message: str):
    output_path = "output_voice.wav"
    generator = tts_pipeline(message, voice="af_heart", speed=1.0, split_pattern=r'\n+')
    for _, _, audio in generator:
        if audio is not None:
            if isinstance(audio, torch.Tensor):
                audio = audio.cpu().numpy()
            audio_data = np.asarray(audio, dtype=np.float32)
            sf.write(output_path, audio_data, 24000)
            break
    return output_path

In [33]:
def chat(history):
    messages: List[Dict[str, Any]] = [{"role": "system", "content": system_message}]
    
    # Process Gradio history format to Ollama string format
    for h in history:
        role = h.get("role")
        content = h.get("content")
        
        if isinstance(content, list):
            text_parts = []
            for item in content:
                if isinstance(item, dict) and "text" in item:
                    text_parts.append(item["text"])
                elif isinstance(item, str):
                    text_parts.append(item)
            content_str = " ".join(text_parts)
        else:
            content_str = str(content) if content is not None else ""
            
        messages.append({"role": role, "content": content_str})
    
    # First LLM Call
    response = ollama.chat(model=LLM_MODEL, messages=messages, tools=tools)
    cities: List[str] = []
    image = None

    # Handle Tool Call loop
    while hasattr(response.message, 'tool_calls') and response.message.tool_calls:
        msg = response.message
        tool_responses, extracted_cities = handle_tool_calls_and_return_cities(msg)
        cities.extend(extracted_cities)
        
        messages.append({
            "role": msg.role,
            "content": msg.content or "",
            "tool_calls": msg.tool_calls
        })
        messages.extend(tool_responses)
        
        response = ollama.chat(model=LLM_MODEL, messages=messages, tools=tools)

    reply = response.message.content or ""
    history.append({"role": "assistant", "content": reply})

    # Trigger Voice and Image
    voice = talker(reply)
    if cities:
        image = artist(cities[0])

    return history, voice, image

In [49]:
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(
        put_message_in_chatbot, 
        inputs=[message, chatbot], 
        outputs=[message, chatbot]
    ).then(
        chat, 
        inputs=chatbot, 
        outputs=[chatbot, audio_output, image_output]
    )

if __name__ == "__main__":
    ui.launch(inbrowser=True, inline=False)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for New York


  0%|          | 0/2 [00:00<?, ?it/s]